## Final evaluation

The 2023 validation set was used for tuning.  
After choosing the ANN and Random Forest configurations, I train them again on 2015-2023 and evaluate them once on 2024-2025.

The test set is not used to choose hyperparameters.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.preprocessing import (
    fit_preprocessor,
    transform_data,
    categorical_features,
    numeric_features
)

from src.evaluation import (
    evaluate_model
)


from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.neighbors import KNeighborsClassifier


from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)


from xgboost import XGBClassifier

from lightgbm import LGBMClassifier


import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Activation
)

from tensorflow.keras import Input

from tensorflow.keras import optimizers


import joblib

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
PROCESSED_DATA_DIR = Path("../data/processed")

RESULTS_DIR = Path("../results")

MODELS_DIR = Path("../models")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

input_file = (PROCESSED_DATA_DIR / "feature_split_matches.csv")

matches = pd.read_csv(input_file, parse_dates=["Date"])

matches = matches.sort_values(
    by=["Date", "MatchID"]
).reset_index(drop=True)

print("Dataset shape:", matches.shape)
print(matches["DataSplit"].value_counts())

Dataset shape: (26536, 53)
DataSplit
Training      18810
Test           5119
Validation     2607
Name: count, dtype: int64


**Create FINAL train and FINAL test**

In [4]:
final_train_data = matches.loc[
    matches["DataSplit"].isin(
        ["Training", "Validation"]
    )
].copy()

test_data = matches.loc[
    matches["DataSplit"] == "Test"
].copy()

In [5]:
# verify final chronological split
print(
    "Final training matches:", 
    len(final_train_data)
)

print("Test matches:", len(test_data))

print(
    "\nFinal training period: ",
    final_train_data["Date"].min(), 
    "-", 
    final_train_data["Date"].max()
)

print(
    "Test period: ", 
    test_data["Date"].min(), 
    "-", 
    test_data["Date"].max()
)

Final training matches: 21417
Test matches: 5119

Final training period:  2015-01-05 00:00:00 - 2023-12-31 00:00:00
Test period:  2024-01-01 00:00:00 - 2025-11-16 00:00:00


In [6]:
assert (
    final_train_data["Date"].max() < test_data["Date"].min()
)

print("Final chronological split is correct.")

Final chronological split is correct.


## Check that all tuning summaries exist and do the final for every model

In [7]:
summary_files = {  
    "ANN":RESULTS_DIR/"ann_cv_summary.csv",
    "Random Forest":RESULTS_DIR/"RF_cv_tuning_summary.csv",
    "k-NN":RESULTS_DIR/"knn_cv_tuning_summary.csv",
    "AdaBoost":RESULTS_DIR/"adaboost_cv_tuning_summary.csv",
    "XGBoost": RESULTS_DIR/ "xgboost_cv_tuning_summary.csv",
    "LightGBM": RESULTS_DIR/"lightgbm_cv_tuning_summary.csv"
}

In [8]:
for model_name, file_path in (summary_files.items()):
    print(model_name, "->", file_path.exists())

ANN -> True
Random Forest -> True
k-NN -> False
AdaBoost -> True
XGBoost -> True
LightGBM -> True


In [11]:
missing_files = [
    str(file_path)
    for file_path in summary_files.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Run the tuning notebook first."
        "Missing files:\n" + "\n".join(missing_files)
    )

print("All tuning summary files found.")

All tuning summary files found.


**Define ANN configurations**
take the configurations tested in the previous notebook

In [12]:
ann_configurations = [
    {
        "name" : "ANN 8 sigmoid",
        "hidden_layers": [8], 
        "activation": "sigmoid", 
        "learning_rate": 0.01, 
        "epochs": 50
    }, 
    
    {
        "name": "ANN 16 relu",
        "hidden_layers": [16], 
        "activation": "relu", 
        "learning_rate": 0.001, 
        "epochs": 50
    },
    
    {
        "name": "ANN 32 relu", 
        "hidden_layers": [32], 
        "activation": "relu", 
        "learning_rate": 0.001, 
        "epochs": 50
    }, 
    
    {
        "name": "ANN 32-16 relu", 
        "hidden_layers": [32, 16], 
        "activation": "relu", 
        "learning_rate": 0.001, 
        "epochs": 50
    }, 
    
    {
        "name": "ANN 32-16 slow", 
        "hidden_layers": [32, 16], 
        "activation": "relu",
        "learning_rate": 0.0005, 
        "epochs": 100
    }
    
]

**Random Forest configurations**

In [13]:
configurations = [
    {
        "name": "RF 100", 
        "n_estimators": 100, 
        "max_depth": None, 
        "min_samples_leaf": 1
    },
    {
        "name": "RF 200", 
        "n_estimators": 200, 
        "max_depth": None, 
        "min_samples_leaf": 1
    },
    {
        "name": "RF 200 depth 10", 
        "n_estimators": 200, 
        "max_depth": 10, 
        "min_samples_leaf": 1
    },
    {
        "name": "RF 200 depth 15", 
        "n_estimators": 200, 
        "max_depth": 15, 
        "min_samples_leaf": 1
    },
    {
        "name": "RF 300 depth 10 leaf 2", 
        "n_estimators": 300, 
        "max_depth": 10, 
        "min_samples_leaf": 2
    },
    {
        "name": "RF 300 depth 15 leaf 2", 
        "n_estimators": 300, 
        "max_depth": 15, 
        "min_samples_leaf": 2
    }
]

**K-NN configurations**

In [14]:
configurations = [
    {
        "name": "kNN k=11",
        "n_neighbors": 11,
        "weights": "uniform"
    },

    {
        "name": "kNN k=31",
        "n_neighbors": 31,
        "weights": "uniform"
    },

    {
        "name": "kNN k=51",
        "n_neighbors": 51,
        "weights": "uniform"
    },

    {
        "name": "kNN k=75",
        "n_neighbors": 75,
        "weights": "uniform"
    },

    {
        "name": "kNN k=101",
        "n_neighbors": 101,
        "weights": "uniform"
    },

    {
        "name": "kNN k=151",
        "n_neighbors": 151,
        "weights": "uniform"
    },

    {
        "name": "kNN k=11 distance",
        "n_neighbors": 11,
        "weights": "distance"
    },

    {
        "name": "kNN k=31 distance",
        "n_neighbors": 31,
        "weights": "distance"
    },

    {
        "name": "kNN k=51 distance",
        "n_neighbors": 51,
        "weights": "distance"
    },

    {
        "name": "kNN k=75 distance",
        "n_neighbors": 75,
        "weights": "distance"
    },

    {
        "name": "kNN k=101 distance",
        "n_neighbors": 101,
        "weights": "distance"
    },

    {
        "name": "kNN k=151",
        "n_neighbors": 151,
        "weights": "distance"
    }
]

**AdaBoost configurations**

In [15]:
configurations = [

    {
        "name": "AdaBoost leaves 2 est 50 lr 0.1",
        "max_leaf_nodes": 2,
        "n_estimators": 50,
        "learning_rate": 0.1
    },

    {
        "name": "AdaBoost leaves 2 est 100 lr 0.1",
        "max_leaf_nodes": 2,
        "n_estimators": 100,
        "learning_rate": 0.1
    },

    {
        "name": "AdaBoost leaves 2 est 50 lr 0.5",
        "max_leaf_nodes": 2,
        "n_estimators": 50,
        "learning_rate": 0.5
    },

    {
        "name": "AdaBoost leaves 2 est 100 lr 0.5",
        "max_leaf_nodes": 2,
        "n_estimators": 100,
        "learning_rate": 0.5
    },

    {
        "name": "AdaBoost leaves 4 est 50 lr 0.1",
        "max_leaf_nodes": 4,
        "n_estimators": 50,
        "learning_rate": 0.1
    },

    {
        "name": "AdaBoost leaves 4 est 100 lr 0.1",
        "max_leaf_nodes": 4,
        "n_estimators": 100,
        "learning_rate": 0.1
    },

    {
        "name": "AdaBoost leaves 4 est 50 lr 0.5",
        "max_leaf_nodes": 4,
        "n_estimators": 50,
        "learning_rate": 0.5
    },

    {
        "name": "AdaBoost leaves 4 est 100 lr 0.5",
        "max_leaf_nodes": 4,
        "n_estimators": 100,
        "learning_rate": 0.5
    }
]

**XGBoost configurations**

In [16]:
configurations = [
    {
        "name": "XGBoost depth 2 est 100 lr 0.05", 
        "n_estimators": 100, 
        "max_depth": 2, 
        "learning_rate": 0.05
    },
    {
        "name": "XGBoost depth 2 est 200 lr 0.05", 
        "n_estimators": 200, 
        "max_depth": 2, 
        "learning_rate": 0.05
    },
    {
        "name": "XGBoost depth 2 est 100 lr 0.10", 
        "n_estimators": 100, 
        "max_depth": 2, 
        "learning_rate": 0.10
    },
    {
        "name": "XGBoost depth 2 est 200 lr 0.10", 
        "n_estimators": 200, 
        "max_depth": 2, 
        "learning_rate": 0.10
    },
    {
        "name": "XGBoost depth 3 est 100 lr 0.05", 
        "n_estimators": 100, 
        "max_depth": 3, 
        "learning_rate": 0.05
    },
    {
        "name": "XGBoost depth 3 est 200 lr 0.05", 
        "n_estimators": 200, 
        "max_depth": 3, 
        "learning_rate": 0.05
    },
    {
        "name": "XGBoost depth 3 est 100 lr 0.10", 
        "n_estimators": 100, 
        "max_depth": 3, 
        "learning_rate": 0.10
    },
    {
        "name": "XGBoost depth 3 est 200 lr 0.10", 
        "n_estimators": 200, 
        "max_depth": 3, 
        "learning_rate": 0.10
    },
    
]

**LightGBM configurations**

In [17]:
configurations = [

    {
        "name": "LightGBM leaves 7 est 100 lr 0.05",
        "num_leaves": 7,
        "n_estimators": 100,
        "learning_rate": 0.05
    },

    {
        "name": "LightGBM leaves 7 est 200 lr 0.05",
        "num_leaves": 7,
        "n_estimators": 200,
        "learning_rate": 0.05
    },

    {
        "name": "LightGBM leaves 7 est 100 lr 0.10",
        "num_leaves": 7,
        "n_estimators": 100,
        "learning_rate": 0.10
    },

    {
        "name": "LightGBM leaves 7 est 200 lr 0.10",
        "num_leaves": 7,
        "n_estimators": 200,
        "learning_rate": 0.10
    },

    {
        "name": "LightGBM leaves 15 est 100 lr 0.05",
        "num_leaves": 15,
        "n_estimators": 100,
        "learning_rate": 0.05
    },

    {
        "name": "LightGBM leaves 15 est 200 lr 0.05",
        "num_leaves": 15,
        "n_estimators": 200,
        "learning_rate": 0.05
    },

    {
        "name": "LightGBM leaves 15 est 100 lr 0.10",
        "num_leaves": 15,
        "n_estimators": 100,
        "learning_rate": 0.10
    },

    {
        "name": "LightGBM leaves 15 est 200 lr 0.10",
        "num_leaves": 15,
        "n_estimators": 200,
        "learning_rate": 0.10
    }
]
